# Actividad 04 - Escalado (StandardScaler ajustado solo en TRAIN)

**REGENERADO 2026-09-07** sobre el nuevo esquema de columnas.

    TRAIN  2016-07..2023-12  n = 90   <- unico conjunto usado para el fit
    VAL    2024-01..2024-12  n = 12   <- solo transform
    TEST   2025-01..2025-12  n = 12   <- solo transform

Se excluyen del escalado `anio`/`mes` (claves) y `mes_sin`/`mes_cos` (ya en [-1,1]).

**Artefactos obsoletos**, registrados en
`resultados_v2_final/pesos/REGISTRO_PRE_REGENERACION.json`:
`scaler_{sutil,dulce}.joblib` (28 features) y `scaler_{sutil,dulce}_v2b_tindex.joblib`
(29 features) corresponden a esquemas de columnas anteriores; se sustituyen por
`scaler_{sutil,dulce}_v2c.joblib`.

In [1]:
import os, sys, json, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings('ignore')
pd.set_option('display.width', 240); pd.set_option('display.max_columns', None)

while not os.path.exists('v2_reentrenamiento/data/processed'):
    os.chdir('..')
RAIZ = os.getcwd()
sys.path.insert(0, os.path.join(RAIZ, 'v2_reentrenamiento', 'src'))
print('Raiz del proyecto:', RAIZ)

PROC      = 'v2_reentrenamiento/data/processed'
INTERIM   = 'v2_reentrenamiento/data/interim'
RAW       = 'v2_reentrenamiento/data/raw'
OUT_R2    = 'v2_reentrenamiento/resultados_v2_final'
PESOS_CSV = f'{OUT_R2}/pesos/pesos_geograficos_train.csv'
CULTIVARES = ['sutil', 'dulce']

Raiz del proyecto: C:\Machine-learming\Machine-Learning-Multimodal--Agro-NLP-Clima-


In [2]:
from features.escalado import EscaladorTrain, N_TRAIN, N_VAL, N_TEST, RANGOS
from features.hashing import Hasher
print('Split:', RANGOS, '| n =', (N_TRAIN, N_VAL, N_TEST))

Split: {'train': ('2016-07', '2023-12'), 'val': ('2024-01', '2024-12'), 'test': ('2025-01', '2025-12')} | n = (90, 12, 12)


## 1. Ajuste sobre train y transformacion de las tres particiones

In [3]:
esc, escalados, params = {}, {}, {}
for c in CULTIVARES:
    df = pd.read_csv(f'{PROC}/master_dataset_{c}_v2_features.csv', encoding='utf-8-sig')
    esc[c] = EscaladorTrain()
    escalados[c] = esc[c].ajustar_transformar(df)
    params[c] = esc[c].parametros()
    p = esc[c].particionar(escalados[c])
    print(f'{c.upper():6s} {escalados[c].shape} | features escaladas={len(esc[c].columnas)} | '
          f'n_samples_seen_={esc[c].scaler.n_samples_seen_}')
    for n in ('train', 'val', 'test'):
        d = p[n]
        print(f'         {n:5s} n={len(d):3d}  {int(d["año"].iloc[0])}-{int(d["mes"].iloc[0]):02d}..'
              f'{int(d["año"].iloc[-1])}-{int(d["mes"].iloc[-1]):02d}')

SUTIL  (114, 48) | features escaladas=44 | n_samples_seen_=90.0
         train n= 90  2016-07..2023-12
         val   n= 12  2024-01..2024-12
         test  n= 12  2025-01..2025-12
DULCE  (114, 48) | features escaladas=44 | n_samples_seen_=90.0
         train n= 90  2016-07..2023-12
         val   n= 12  2024-01..2024-12
         test  n= 12  2025-01..2025-12


## 2. Verificacion del escalado

In [4]:
for c in CULTIVARES:
    v = esc[c].verificar(escalados[c])
    print(f'{c.upper():6s} |media_train| max={v["train_abs_media_max"]:.2e}  '
          f'std_train in [{v["train_std_min"]:.6f},{v["train_std_max"]:.6f}]  '
          f'|media_val| max={v["val_abs_media_max"]:.4f}  |media_test| max={v["test_abs_media_max"]:.4f}')
    print(f'         NaN={v["nan_total"]}  Inf={v["inf_total"]}  '
          f'no escaladas={v["columnas_no_escaladas"]}  fit_solo_train_ok={v["fit_solo_train_ok"]}')
    assert v['fit_solo_train_ok'] and v['nan_total'] == 0 and v['inf_total'] == 0

SUTIL  |media_train| max=4.91e-15  std_train in [1.000000,1.000000]  |media_val| max=1.9631  |media_test| max=2.4250
         NaN=0  Inf=0  no escaladas=['año', 'mes', 'mes_sin', 'mes_cos']  fit_solo_train_ok=True
DULCE  |media_train| max=5.67e-15  std_train in [1.000000,1.000000]  |media_val| max=2.2481  |media_test| max=2.4250
         NaN=0  Inf=0  no escaladas=['año', 'mes', 'mes_sin', 'mes_cos']  fit_solo_train_ok=True


## 3. Persistencia de datasets, scalers y parametros

In [5]:
hashes = {}
for c in CULTIVARES:
    r_csv = f'{PROC}/master_dataset_{c}_v2_escalado.csv'
    escalados[c].to_csv(r_csv, index=False, encoding='utf-8-sig')
    r_scl = f'{OUT_R2}/scalers/scaler_{c}_v2c.joblib'
    esc[c].guardar(r_scl)
    r_par = f'{OUT_R2}/scalers/scaler_{c}_v2c_parametros.csv'
    params[c].to_csv(r_par, index=False, encoding='utf-8-sig')
    hashes[c] = {'maestro':    Hasher.describir(f'{PROC}/master_dataset_{c}_v2.csv'),
                 'features':   Hasher.describir(f'{PROC}/master_dataset_{c}_v2_features.csv'),
                 'escalado':   Hasher.describir(r_csv),
                 'scaler':     Hasher.describir(r_scl),
                 'parametros': Hasher.describir(r_par)}
    print(f'{c.upper():6s} guardado escalado + scaler + parametros')
    for k, h in hashes[c].items():
        print(f'         {k:11s} {h["sha256"]}')

SUTIL  guardado escalado + scaler + parametros
         maestro     220d9a9aabf6e41a31ee5950223eab3a74ddeadba18b8a8e281021b2af84b3f8
         features    a6f23d0eb3178514c75f6851f3c2039837a192acc5f2b008bc445dda35968cc9
         escalado    d664fd728798d52e5c1bbe702745d40d49b8761513cef148e0a56f2481ca7941
         scaler      ba51666f00ded716c63698dd04e29068d58d3cab064202384bfbd4435fbf350e
         parametros  4e4d0e1fa6555d605e71cf9c378c4a33e6a8a6b0d238024be6d447f813fee80e
DULCE  guardado escalado + scaler + parametros
         maestro     547443d1a117217b8f01f723f345749d530597b45210922a7e609469521934d0
         features    c0e7cbe99fdf514cf3bde3c3f735b7d14fc63b27c7f3f163ec665ae25696a188
         escalado    38466a4f9581f7d8058962248161d7cb1d467a82e7c2092de73364920dd84f80
         scaler      1bf919b4c00d236801f12df420b79a16f47c4fe4df3b711472cdc36428eb962b
         parametros  e62e44ba9e60c79ed4f72b242fe0c90e7ade5648613b6025f61e4a8b2aac3f64


In [6]:
from datetime import datetime
meta = {'artefacto': 'fase2_v2_regenerada',
        'fecha_generacion': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
        'convencion_sha256': 'NUEVA en v2 (D19). No existia en v1 ni en la Fase 2 v2 original.',
        'split': RANGOS,
        'n': {'train': N_TRAIN, 'val': N_VAL, 'test': N_TEST},
        'scaler': {c: {'n_features_in_': int(esc[c].scaler.n_features_in_),
                       'n_samples_seen_': int(esc[c].scaler.n_samples_seen_),
                       'columnas_escaladas': esc[c].columnas,
                       'columnas_no_escaladas': ['anio', 'mes', 'mes_sin', 'mes_cos']}
                   for c in CULTIVARES},
        'hashes': hashes}
r = f'{OUT_R2}/pesos/fase2_v2_regenerada_meta.json'
json.dump(meta, open(r, 'w', encoding='utf-8'), indent=2, ensure_ascii=False)
print('Guardado:', r)

Guardado: v2_reentrenamiento/resultados_v2_final/pesos/fase2_v2_regenerada_meta.json
